In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

import pandas as pd

from positionnal_encoding import PositionalEncoding
from CSLR_transformer import CSLRTransformer
from hyperparameters import BATCH_SIZE, SEQ_LEN, INPUT_DIM, DEVICE, MAX_TARGET_LEN, VOCAB_SIZE
from model import model, optimizer, ctc_loss

# ===== Fake BERT / MaskGIT embeddings --> Shape is like this --> [B = batch size,T = longueur temporelle ,D = dimension des embeddings]
# this is a mock embeddings because I wanted to see if this will work.
# need to be removed

embeddings = torch.randn(
    BATCH_SIZE,
    SEQ_LEN,
    INPUT_DIM
).to(DEVICE)

# ====Fake target gloss sequences========== (same as before)

target_lengths = torch.randint( #### NEED TO CHANGE THINGS HERE
    low=3,
    high=MAX_TARGET_LEN,
    size=(BATCH_SIZE,)
)

### need to remove the thing between green commentary

targets = []

for length in target_lengths:
    t = torch.randint( ### NEED TO CHANGE THINGS HERE
        0,
        VOCAB_SIZE,
        (length,)
    )
    targets.append(t)

targets = torch.cat(targets).to(DEVICE)

input_lengths = torch.full(
    (BATCH_SIZE,),
    SEQ_LEN,
    dtype=torch.long
).to(DEVICE)

target_lengths = target_lengths.to(DEVICE)

# ==== Forward ========== 

logits = model(embeddings)

print("Logits shape:", logits.shape)

# ==== CTC expects [T,B,C] --> T = temporalité, B = batch size and C = number of class --> vocab + 1 (1=blank space) =====

log_probs = F.log_softmax(
    logits,
    dim=-1
)

log_probs = log_probs.transpose(
    0,
    1
)

loss = ctc_loss(
    log_probs,
    targets,
    input_lengths,
    target_lengths
)

# ==== Backprop =========================

optimizer.zero_grad()

loss.backward()

optimizer.step()

print(f"CTC Loss: {loss.item():.4f}")

# ==== Greedy decoding example ===============

predictions = logits.argmax(dim=-1)

print("\nPrediction shape:")
print(predictions.shape)

print("\nFirst sample raw prediction:")
print(predictions[0][:30])

Logits shape: torch.Size([4, 120, 401])
CTC Loss: 68.7569

Prediction shape:
torch.Size([4, 120])

First sample raw prediction:
tensor([310,  91,  80, 200,  91,  91,  16,  80, 310, 230,  83,  16, 114, 230,
        171,  16, 171, 288, 288, 230, 230, 114, 114, 114, 310, 288,  80, 310,
        310, 307])
